# 03 — [SENİN ALGORİTMANIN ADI] — Baseline + Tuning

**Bu bir ŞABLON dosyadır.** Aşağıdaki adımları izle:

1. Bu dosyayı kopyala → `03_[algoritmanin_adi].ipynb` olarak yeniden adlandır (örn. `03_knn.ipynb`, `03_random_forest.ipynb`)
2. `>>> ... <<<` etiketleri olan yerleri kendi algoritmanla doldur
3. Kernel → Restart & Run All ile çalıştır
4. Bölüm 5'te kendi yorumunu yaz (markdown)
5. Branch aç → push → PR

**Önemli:** 
- Tüm üyeler aynı `data/X_train_final.csv` ve `X_test_final.csv` dosyalarını kullanıyor — sonuçlar **bire bir karşılaştırılabilir**.
- `random_state=42` sabit. **Değiştirme.**
- Preprocessing'i tekrar yapma — veri zaten ölçeklenmiş + encode edilmiş + feature engineering yapılmış.

## 1. Veri Yükleme (HERKES AYNI — DEĞİŞTİRME)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, ConfusionMatrixDisplay,
                              classification_report)
from sklearn.model_selection import GridSearchCV

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

RANDOM_STATE = 42

X_train = pd.read_csv('../data/X_train_final.csv')
X_test  = pd.read_csv('../data/X_test_final.csv')
y_train = pd.read_csv('../data/y_train_final.csv').iloc[:, 0]
y_test  = pd.read_csv('../data/y_test_final.csv').iloc[:, 0]

siniflar = ['Ekonomik', 'Orta', 'Yüksek', 'Premium']

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'\nSınıf dağılımı (train):')
print(y_train.value_counts(normalize=True).round(3).to_string())

## 2. Baseline — Default Parametrelerle

Algoritmanı **hyperparameter tuning yapmadan**, sklearn'in varsayılan ayarlarıyla çalıştır. Bu "zemin skor".

**Yapacakların:**
1. `>>> import <<<` satırını aç (kendi algoritmanın import'u)
2. `>>> baseline_model = ... <<<` satırına kendi sınıfını yaz (örn. `KNeighborsClassifier()`)
3. Geri kalan kod aynen çalışır

In [ ]:
# >>> SENİN ALGORİTMANIN IMPORT'U <<<
# Örnek: from sklearn.neighbors import KNeighborsClassifier
# Örnek: from sklearn.tree import DecisionTreeClassifier
# Örnek: from sklearn.naive_bayes import GaussianNB
# ...

# >>> BASELINE MODEL — default parametreler <<<
# Örnek: baseline_model = KNeighborsClassifier()
# Örnek: baseline_model = DecisionTreeClassifier(criterion='entropy', random_state=RANDOM_STATE)
baseline_model = None  # ← BURAYI DEĞİŞTİR

# Eğit + tahmin et (HERKES AYNI)
baseline_model.fit(X_train, y_train)
y_pred_base = baseline_model.predict(X_test)

# Metrikleri yazdır (HERKES AYNI)
print('=== BASELINE — Default Parametreler ===\n')
print(f'Accuracy : {accuracy_score(y_test, y_pred_base):.4f}')
print(f'F1 macro : {f1_score(y_test, y_pred_base, average="macro"):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_base, average="macro"):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_base, average="macro"):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_base, target_names=siniflar))

## 3. Tuning — GridSearchCV ile Optimal Parametre Arama

**Yapacakların:**
1. `docs/TUNING_REHBERI.md`'de **kendi algoritmana ait bölümü** oku — hangi parametreleri tune etmen gerektiği yazıyor
2. **Her parametre için kendin uygun değerler bul** — sklearn dokümanı, ders slaytları
3. Aşağıdaki `param_grid = {...}` içine kendi seçtiğin değerleri yaz
4. Çalıştır (1-10 dakika)

**Bölüm 5'te yazacağın yorumda mutlaka anlat:** Niçin bu değerleri seçtin? Algoritmanı tanıdığın için ne tür değerlerin işe yarayacağını biliyordun.

**Sabit ayarlar (değiştirme):** `cv=5` (5-fold cross-validation), `scoring='f1_macro'`.

In [ ]:
# >>> SENİN ALGORİTMANIN PARAMETRE GRİDİ <<<
# docs/TUNING_REHBERI.md'de hangi parametreleri tune edeceğin yazıyor.
# DEĞERLERİ KENDİN ARAŞTIRIP YAZ — sklearn dokümanına ve ders notlarına bak.
param_grid = {
    # 'parametre_1': [...],   # ← Sen doldur
    # 'parametre_2': [...],
}

# GridSearchCV (HERKES AYNI — değiştirme)
grid = GridSearchCV(
    baseline_model,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
grid.fit(X_train, y_train)

print(f'En iyi parametreler : {grid.best_params_}')
print(f'CV F1-macro skoru   : {grid.best_score_:.4f}')

# Optimize edilmiş model ile tahmin
best_model = grid.best_estimator_
y_pred_tuned = best_model.predict(X_test)

print('\n=== TUNED — Optimal Parametreler ===\n')
print(f'Accuracy : {accuracy_score(y_test, y_pred_tuned):.4f}')
print(f'F1 macro : {f1_score(y_test, y_pred_tuned, average="macro"):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_tuned, average="macro"):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred_tuned, average="macro"):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_tuned, target_names=siniflar))

## 4. Karşılaştırma — Baseline vs Tuned (HERKES AYNI)

In [ ]:
# Tablo: baseline vs tuned
karsilastirma = pd.DataFrame({
    'Metrik': ['Accuracy', 'F1-macro', 'Precision (macro)', 'Recall (macro)'],
    'Baseline': [
        accuracy_score(y_test, y_pred_base),
        f1_score(y_test, y_pred_base, average='macro'),
        precision_score(y_test, y_pred_base, average='macro'),
        recall_score(y_test, y_pred_base, average='macro'),
    ],
    'Tuned': [
        accuracy_score(y_test, y_pred_tuned),
        f1_score(y_test, y_pred_tuned, average='macro'),
        precision_score(y_test, y_pred_tuned, average='macro'),
        recall_score(y_test, y_pred_tuned, average='macro'),
    ],
})
karsilastirma['Δ (iyileşme)'] = (karsilastirma['Tuned'] - karsilastirma['Baseline']).round(4)
karsilastirma[['Baseline', 'Tuned']] = karsilastirma[['Baseline', 'Tuned']].round(4)
print('Baseline vs Tuned karşılaştırma:\n')
print(karsilastirma.to_string(index=False))

In [ ]:
# Bar chart — metrik bazlı baseline vs tuned
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(karsilastirma))
width = 0.35
ax.bar(x - width/2, karsilastirma['Baseline'], width, label='Baseline', color='#4c72b0')
ax.bar(x + width/2, karsilastirma['Tuned'],    width, label='Tuned',    color='#55a868')
ax.set_xticks(x)
ax.set_xticklabels(karsilastirma['Metrik'])
ax.set_ylabel('Skor')
ax.set_title('Baseline vs Tuned — Metrik Karşılaştırması')
ax.legend()
ax.set_ylim(0, 1)
for i, (b, t) in enumerate(zip(karsilastirma['Baseline'], karsilastirma['Tuned'])):
    ax.text(i - width/2, b + 0.01, f'{b:.3f}', ha='center', fontsize=9)
    ax.text(i + width/2, t + 0.01, f'{t:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# İki confusion matrix yan yana
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, y_pred, baslik in zip(axes, [y_pred_base, y_pred_tuned], ['Baseline', 'Tuned']):
    cm = confusion_matrix(y_test, y_pred, labels=siniflar)
    disp = ConfusionMatrixDisplay(cm, display_labels=siniflar)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(baslik)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (sadece ağaç-tabanlı modeller için)
# Eğer algoritman karar ağacı / random forest ise aşağıdaki kodu çalıştır.
# Diğer algoritmalarda (k-NN, NB, K-Means) bu hücre hata verebilir, atla.

if hasattr(best_model, 'feature_importances_'):
    fi = pd.DataFrame({
        'feature': X_train.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False).head(15)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=fi, y='feature', x='importance', palette='viridis')
    plt.title('En önemli 15 özellik (tuned model)')
    plt.tight_layout()
    plt.show()
else:
    print('Bu algoritmada feature_importances_ yok — bu hücreyi atla.')

---
## 5. Yorum — Kendi Algoritmanı Anlat

**Bu bölüm raporun temelini oluşturur. Aşağıdaki 4 alt bölümü kendi cümlelerinle doldur.**

### 5.1 Algoritmanın çalışma mantığı

> Derste hocanın anlattığı şekilde algoritmayı 1-2 paragrafla özetle. Örnekler:
> - **k-NN:** "Yeni veri için en yakın k komşunun çoğunluk sınıfını seçer. Mesafe metriği olarak Euclidean..."
> - **ID3:** "Information Gain'e göre en bilgi verici özelliği seç, ona göre dalla..."
> - **Naive Bayes:** "Bayes teoremi + özellikler arası bağımsızlık varsayımı..."

[BURAYA YAZ]

### 5.2 Tuning sonuçları

> 1-2 paragraf. Sorular:
> - Hangi parametreler optimal çıktı? (`grid.best_params_` çıktısına bak)
> - Niye bu parametrelerle iyileşti? (Algoritma bilgini kullanarak yorumla)
> - Baseline → Tuned arasında F1-macro kaç puan iyileşti?

[BURAYA YAZ]

### 5.3 Sınıf bazlı performans

> Confusion matrix'e bak. Sorular:
> - Hangi sınıfı en iyi tahmin etti? (Köşegende en yüksek değer)
> - En çok hangi 2 sınıfı karıştırdı? (Köşegen dışında en yüksek değer)
> - Bu karışıklığın sebebi ne olabilir?

[BURAYA YAZ]

### 5.4 Bu veriye uygun mu?

> 1 paragraf. Sorular:
> - Algoritma SUV fiyat sınıflandırması için iyi/kötü mü?
> - Veri özelliklerinden hangileri bu algoritmaya uygun? (Çok özellik var → ağaçlar iyi, doğrusal sınır var → LR iyi, vs.)
> - Geliştirilebilir miydi? (Daha fazla feature engineering, farklı encoding, ensembles vb.)

[BURAYA YAZ]

---

## Bitirdikten Sonra

1. Notebook'un tamamını **Kernel → Restart & Run All** ile çalıştır → hata olmadığından emin ol
2. Branch aç:
   ```bash
   git checkout -b feature/[algoritma-adin]
   git add notebooks/03_[algoritma-adin].ipynb
   git commit -m "[Algoritma] tuning ekledi, F1: 0.XX"
   git push origin feature/[algoritma-adin]
   ```
3. GitHub'da Pull Request aç → liderin onaylamasını bekle